In [1]:
import pandas as pd

In [2]:
# Read OxCGRT Australia data
OxCGRT = pd.read_csv("../Raw data/OxCGRT_AUS_latest.csv", low_memory=False)

OxCGRT.shape

(9864, 61)

In [3]:
# Remove incomplete policy variables
analysis_data = OxCGRT.copy()
candidate_policy_variables = OxCGRT.columns[6:49].tolist()

missing_summary = pd.DataFrame({
    "missing_count": analysis_data[candidate_policy_variables].isna().sum()
    }).sort_values("missing_count", ascending=False)

display(missing_summary)

,missing_count
M1_Wildcard,9864
E3_Fiscal measures,9170
E4_International support,9169
H4_Emergency investment in healthcare,9167
H5_Investment in vaccines,8453
C6M_Flag,8345
V2C_Vaccine age eligibility/availability age floor (at risk summary),3995
V2B_Vaccine age eligibility/availability age floor (general population summary),3995
C4M_Flag,3947
C5M_Flag,3868


In [4]:
complete_policy_variables = missing_summary.index[missing_summary["missing_count"] == 0].tolist()

display(complete_policy_variables)

['H3_Contact tracing',
 'V2A_Vaccine Availability (summary)',
 'V3_Vaccine Financial Support (summary)',
 'C2M_Workplace closing',
 'C3M_Cancel public events',
 'C4M_Restrictions on gatherings',
 'C5M_Close public transport',
 'C6M_Stay at home requirements',
 'C7M_Restrictions on internal movement',
 'V1_Vaccine Prioritisation (summary)',
 'H2_Testing policy',
 'C8EV_International travel controls',
 'E1_Income support',
 'H8M_Protection of elderly people',
 'E2_Debt/contract relief',
 'H7_Vaccination policy',
 'H1_Public information campaigns',
 'H6M_Facial Coverings',
 'C1M_School closing']

In [5]:
# Examine the correlation between policy variables
correlation_matrix = analysis_data[complete_policy_variables].corr(method="spearman")

high_correlation_pairs = []
for row_index, first_variable in enumerate(complete_policy_variables):
    for second_variable in complete_policy_variables[row_index + 1:]:
        correlation = correlation_matrix.loc[first_variable, second_variable]
        if abs(correlation) >= 0.75:
            high_correlation_pairs.append({
                "variable_1": first_variable,
                "variable_2": second_variable,
                "spearman_correlation": round(correlation, 4)
            })

high_correlation_pairs = pd.DataFrame(high_correlation_pairs).sort_values(
    "spearman_correlation", ascending=False
)
display(high_correlation_pairs)


,variable_1,variable_2,spearman_correlation
2,V2A_Vaccine Availability (summary),H7_Vaccination policy,0.9857
7,V1_Vaccine Prioritisation (summary),H7_Vaccination policy,0.9653
1,V2A_Vaccine Availability (summary),V1_Vaccine Prioritisation (summary),0.9641
4,V3_Vaccine Financial Support (summary),H7_Vaccination policy,0.9117
0,V2A_Vaccine Availability (summary),V3_Vaccine Financial Support (summary),0.9009
3,V3_Vaccine Financial Support (summary),V1_Vaccine Prioritisation (summary),0.8737
6,C3M_Cancel public events,C4M_Restrictions on gatherings,0.8043
5,C2M_Workplace closing,C3M_Cancel public events,0.7772


In [6]:
# C3M is highly correlated with both C2M and C4M.
# H7 is highly correlated with V1-V4 vaccine policy variables.
redundant_policy_variables = [
    "C3M_Cancel public events",
    "V1_Vaccine Prioritisation (summary)",
    "V2A_Vaccine Availability (summary)",
    "V3_Vaccine Financial Support (summary)"
]

selected_policy_variables = [
    'H3_Contact tracing',
    'C2M_Workplace closing',
    'C4M_Restrictions on gatherings',
    'C5M_Close public transport',
    'C6M_Stay at home requirements',
    'C7M_Restrictions on internal movement',
    'H2_Testing policy',
    'C8EV_International travel controls',
    'E1_Income support',
    'H8M_Protection of elderly people',
    'E2_Debt/contract relief',
    'H1_Public information campaigns',
    'H6M_Facial Coverings',
    'C1M_School closing',
    'H7_Vaccination policy'
]

In [7]:
# Use the policy variables retained by the previous selection steps
policy_variables = selected_policy_variables.copy()

policy_data = OxCGRT[["RegionName", "Date"] + policy_variables].copy()
policy_data = policy_data.rename(columns={"RegionName": "state"})

policy_data.head()

,state,Date,H3_Contact tracing,C2M_Workplace closing,C4M_Restrictions on gatherings,C5M_Close public transport,C6M_Stay at home requirements,C7M_Restrictions on internal movement,H2_Testing policy,C8EV_International travel controls,E1_Income support,H8M_Protection of elderly people,E2_Debt/contract relief,H1_Public information campaigns,H6M_Facial Coverings,C1M_School closing,H7_Vaccination policy
0,Australian Capital Territory,20200101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Australian Capital Territory,20200102,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Australian Capital Territory,20200103,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Australian Capital Territory,20200104,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Australian Capital Territory,20200105,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
# Convert date and sort observations by state and date
policy_data["Date"] = pd.to_datetime(policy_data["Date"].astype(str),format="%Y%m%d",errors="coerce")

policy_data = policy_data.sort_values(["state", "Date"]).reset_index(drop=True)

policy_data.head()

,state,Date,H3_Contact tracing,C2M_Workplace closing,C4M_Restrictions on gatherings,C5M_Close public transport,C6M_Stay at home requirements,C7M_Restrictions on internal movement,H2_Testing policy,C8EV_International travel controls,E1_Income support,H8M_Protection of elderly people,E2_Debt/contract relief,H1_Public information campaigns,H6M_Facial Coverings,C1M_School closing,H7_Vaccination policy
0,Australian Capital Territory,2020-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Australian Capital Territory,2020-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Australian Capital Territory,2020-01-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Australian Capital Territory,2020-01-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Australian Capital Territory,2020-01-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
policy_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9864 entries, 0 to 9863
Data columns (total 17 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                                 --------------  -----         
 0   state                                  8768 non-null   object        
 1   Date                                   9864 non-null   datetime64[ns]
 2   H3_Contact tracing                     9864 non-null   float64       
 3   C2M_Workplace closing                  9864 non-null   float64       
 4   C4M_Restrictions on gatherings         9864 non-null   float64       
 5   C5M_Close public transport             9864 non-null   float64       
 6   C6M_Stay at home requirements          9864 non-null   float64       
 7   C7M_Restrictions on internal movement  9864 non-null   float64       
 8   H2_Testing policy                      9864 non-null   float64       
 9   C8EV_International travel controls     9864 non-null   float64 

In [10]:
# Remove rows with missing values
policy_data = policy_data.dropna().reset_index(drop=True)

In [11]:
# Scale all policy variables to a common 0-5 range
policy_maximums = {
    "C1M_School closing": 3,
    "C2M_Workplace closing": 3,
    "C4M_Restrictions on gatherings": 4,
    "C5M_Close public transport": 2,
    "C6M_Stay at home requirements": 3,
    "C7M_Restrictions on internal movement": 2,
    "C8EV_International travel controls": 4,
    "E1_Income support": 2,
    "E2_Debt/contract relief": 2,
    "H1_Public information campaigns": 2,
    "H2_Testing policy": 3,
    "H3_Contact tracing": 2,
    "H6M_Facial Coverings": 4,
    "H7_Vaccination policy": 5,
    "H8M_Protection of elderly people": 3
}

for variable, maximum in policy_maximums.items():
    policy_data[variable] = (policy_data[variable] / maximum * 5)

In [12]:
# Calculate the mean policy intensity for each policy category
C_variables = [variable for variable in policy_variables if variable.startswith("C")]
E_variables = [variable for variable in policy_variables if variable.startswith("E")]
H_variables = [variable for variable in policy_variables if variable.startswith("H")]

policy_data["C_policy_mean"] = policy_data[C_variables].mean(axis=1).round(4)
policy_data["E_policy_mean"] = policy_data[E_variables].mean(axis=1).round(4)
policy_data["H_policy_mean"] = policy_data[H_variables].mean(axis=1).round(4)

# Calculate the overall policy intensity
policy_data["overall_policy_intensity"] = policy_data[["C_policy_mean", "E_policy_mean", "H_policy_mean"]].mean(axis=1).round(4)

policy_data[["C_policy_mean", "E_policy_mean", "H_policy_mean"]+ ["overall_policy_intensity"]].describe()

,C_policy_mean,E_policy_mean,H_policy_mean,overall_policy_intensity
count,8768.000000,8768.000000,8768.000000,8768.000000
mean,1.923598,2.460795,3.299134,2.561176
std,1.193155,1.419912,0.960847,0.938601
min,0.000000,0.000000,0.000000,0.000000
25%,0.654800,1.250000,2.916700,2.070100
50%,2.083300,2.500000,3.527800,2.652100
75%,2.678600,3.750000,4.000000,3.247400
max,4.761900,5.000000,4.722200,4.371700


In [13]:
# Calculate cut-off points that create approximately equal-sized groups
lower_cutoff = policy_data["overall_policy_intensity"].quantile(1 / 3)
upper_cutoff = policy_data["overall_policy_intensity"].quantile(2 / 3)

print("Low/medium cutoff:", lower_cutoff)
print("Medium/high cutoff:", upper_cutoff)


Low/medium cutoff: 2.285
Medium/high cutoff: 3.0489


In [14]:
# Categorise overall policy intensity into low, medium, and high levels
def categorise_policy_intensity(value):
    if value < lower_cutoff:
        return "low"
    elif value < upper_cutoff:
        return "medium"
    else:
        return "high"

policy_data["policy_intensity_level"] = policy_data["overall_policy_intensity"].apply(
    categorise_policy_intensity
)

policy_intensity_counts = policy_data["policy_intensity_level"].value_counts()
display(policy_intensity_counts)

policy_intensity_level
high      2997
low       2920
medium    2851
Name: count, dtype: int64

In [15]:
# Keep only variables required for the final policy dataset
policy_data = policy_data[[
    "state",
    "Date",
    "C_policy_mean",
    "E_policy_mean",
    "H_policy_mean",
    "overall_policy_intensity",
    "policy_intensity_level"
]].copy()

policy_data.iloc[100:105]

,state,Date,C_policy_mean,E_policy_mean,H_policy_mean,overall_policy_intensity,policy_intensity_level
100,Australian Capital Territory,2020-04-10,4.4048,3.75,2.7778,3.6442,high
101,Australian Capital Territory,2020-04-11,4.4048,3.75,2.7778,3.6442,high
102,Australian Capital Territory,2020-04-12,4.4048,3.75,2.7778,3.6442,high
103,Australian Capital Territory,2020-04-13,4.4048,3.75,2.7778,3.6442,high
104,Australian Capital Territory,2020-04-14,4.4048,3.75,2.7778,3.6442,high


In [16]:
policy_data.shape

(8768, 7)

In [17]:
# Save the cleaned policy data
policy_data.to_csv("../Cleaned data/cleaned_OxCGRT_policy_data.csv", index=False, date_format="%Y-%m-%d")